# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a walkthrough for loading and exploring the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library.

This dataset contains anonymized survey data regarding mental health indicators collected via GAD-7, PHQ-9, and PSQ scales, including demographic information, useful for research and public health initiatives in Africa.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant metadata URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Title: {metadata.name}\n\nDescription: {metadata.description}\n")print(f"Cite as: {metadata.citeAs}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll inspect which record sets exist and what their field structure is. **All entities are referenced by their `@id`.**

In [ ]:
print("Available record sets (by @id):")
record_sets = [rs['@id'] for rs in metadata.to_json().get('recordSet', [])]
for idx, rs_id in enumerate(record_sets):
    print(f"{idx+1}. {rs_id}")

if not record_sets:
    print("No top-level record set references; will attempt to infer from available data files...")

# For Croissant datasets, you can also access record sets directly via the API
if hasattr(dataset, "list_record_sets"):
    print("\nRecord sets by mlcroissant (by @id):")
    for rec in dataset.list_record_sets():
        print(f"- {rec['@id']} : {rec.get('name','')}")

Let's explore all record sets with their respective fields and their `@id`s. This is helpful to know what data columns are available for extraction and analysis.

In [ ]:
def print_record_sets_fields(ds):
    json_md = ds.metadata.to_json()
    record_sets = json_md.get('recordSet', [])
    if isinstance(record_sets, dict):
        record_sets = [record_sets]
    for rs in record_sets:
        rs_id = rs.get('@id', 'NO_ID')
        print(f"\nRecord set @id: {rs_id}")
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        print("Fields in this record set:")
        for f in fields:
            print(f"  - {f.get('@id','NO_ID')} | {f.get('name','NO_NAME')}")
        print()

print_record_sets_fields(dataset)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

For this dataset, let's use the main survey record set. We'll reference its unique `@id`.

In [ ]:
# Most mlcroissant-compatible datasets have a main record set named something like 'kilifi_survey' or similar.
# We'll auto-detect. You can use the above printout to choose a specific @id.
from pprint import pprint

# Extract available record set IDs
rs_json = dataset.metadata.to_json().get('recordSet', [])
if isinstance(rs_json, dict):
    rs_json = [rs_json]
record_set_ids = [rs['@id'] for rs in rs_json]

# Try each record set, print first 2 records for inspection
dfs = {}
for rec_id in record_set_ids:
    try:
        print(f"\nReading record set: {rec_id}")
        records = list(dataset.records(record_set=rec_id))
        df = pd.DataFrame(records)
        dfs[rec_id] = df
        print(f"Shape: {df.shape}")
        print(f"Columns: {df.columns.tolist()}")
        pprint(df.head(2).to_dict())
    except Exception as e:
        print(f"Could not load {rec_id}: {e}")

if not dfs:
    raise RuntimeError("No usable record sets found!")

# Pick the main record set (choose the first for this example)
main_record_set_id = list(dfs.keys())[0]
print(f"\nChosen main record set for further analysis: {main_record_set_id}\n")

# Preview all columns for main record set
print(f"Columns in main record set (@id = {main_record_set_id}):")
print(dfs[main_record_set_id].columns.tolist())
dfs[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Let's use the GAD-7 (anxiety) score, often named `gad7_score` or similar, as our numeric field for transformations. We'll reference by the detected column `@id` (from the previous outputs).

In [ ]:
df = dfs[main_record_set_id].copy()

# Try to get a GAD-7 field from the available columns
possible_gad7_cols = [c for c in df.columns if c.lower().startswith('gad7') or 'anxi' in c.lower()]
if possible_gad7_cols:
    numeric_field = possible_gad7_cols[0]  # choose first candidate
else:
    numeric_field = df.select_dtypes(include='number').columns[0]  # fallback to first numeric field

print(f"Using numeric field for analysis: {numeric_field}")

# Remove rows with missing values for this field
df_valid = df[df[numeric_field].notnull()]

# Filter records above a threshold
threshold = 10
filtered_df = df_valid[df_valid[numeric_field] > threshold]
print(f"\nFiltered records with {numeric_field} > {threshold} (N={filtered_df.shape[0]}):")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"\nNormalized '{numeric_field}' for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a categorical field (e.g., gender or education); pick if available
group_candidates = [c for c in df.columns if any(x in c.lower() for x in ['gender','sex','education','village'])]
if group_candidates:
    group_field = group_candidates[0]
    print(f"\nGrouping by field: {group_field}")
    grouped_df = (
        filtered_df.groupby(group_field)[numeric_field].mean().to_frame(name=f"mean_{numeric_field}")
    )
    print(grouped_df.head())
else:
    print("No suitable categorical field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset, e.g., GAD-7 or PHQ-9 score distributions, or scores by education/gender group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot for the main numeric field
plt.figure(figsize=(8,5))
sns.histplot(df_valid[numeric_field], kde=True, bins=15)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

(group_candidates and filtered_df.shape[0] > 0)
if group_candidates:
    group_field = group_candidates[0]
    plt.figure(figsize=(10,5))
    sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field])
    plt.title(f"{numeric_field} by {group_field}")
    plt.show()

## 6. Conclusion
This notebook demonstrated how to use `mlcroissant` for seamless loading and exploration of a Croissant-structured dataset. Key findings:
- The dataset includes well-described mental health and demographic survey data from Kilifi, Kenya.
- Scores such as GAD-7 can be easily selected and statistically analyzed using standard Python tools.
- Advanced analyses and data splits are feasible using entity `@id` references from the metadata.

For further use, consult the [mlcroissant documentation](https://mlcommons.org/croissant/) and refer to the full Croissant schema referenced above.